# 참여 ③ · AI 답변 검증 — 새 자료에서도 맞는가 (9주차 · 2026. 11. 9.)

기초지질데이터과학 · **제출: 오늘 23:59, LMS 참여 ③ — 이 노트북(.ipynb) 하나, 각자 제출** (조 반박문은 조원끼리 같은 문장이어도 됩니다)

**조 번호:**　　　**조원:**　　　　　　　　**이름 · 학번:**　　　　　　**내가 맡은 일 (한 줄):**

- 우리 조 문제의 CSV 하나를 왼쪽 📁 파일 창에 끌어다 놓으세요. **문제 1** `porosity_depth.csv` · **문제 2** `velocity_density.csv` · **문제 3** `sandstone_features.csv`. 셋 다 **교육용 합성 자료**입니다(자료 설명은 배포물 뒷면).
- 순서: ① 배포물의 **프롬프트를 그대로** AI에게 보내고(CSV 첨부 또는 내용 붙여넣기) 답을 기록 → ② 아래 도구로 **같은 기준에서** 검증 → ③ 반박문 → ④ 기록지.
- **참여 ③ 인정 기준**: 반박문에 ⓐ AI 답의 핵심 문장 인용 ⓑ 이 노트북의 수치 근거 1개 이상 ⓒ 강의(쪽수)나 지질학적 근거 1개 ⓓ 대안. AI가 맞게 답했다면 같은 틀로 **"검증 보고"**를 써도 인정됩니다 — 핵심은 AI가 틀렸는지가 아니라, **AI 밖의 근거로 확인했는가**입니다.

## 0. 도구 — 실행만 하세요 (Shift + Enter)

회귀 시간(289–306쪽)에 배운 최소자승을 그대로 씁니다. `G_poly`·`G_cols`가 설계 행렬 **G**를, `fit`이 **m = (GᵀG)⁻¹Gᵀd**를 만듭니다. 새로 생긴 것은 하나뿐 — **`cv_predict`: 자료 일부를 빼 두고 나머지로만 맞춘 뒤, 빼 둔 자료를 예측합니다.** 이렇게 잰 오차가 **검증 오차**입니다.

| 함수 | 하는 일 |
|---|---|
| `loo(n)` | LOOCV — 하나씩 빼기 |
| `kfold(n, k)` | k-겹 — 무작위 k묶음 |
| `deepest(x, k)` | x가 가장 큰 k개를 한꺼번에 빼기 (외삽을 흉내) |
| `report(이름, G 만드는 함수, d, 나누기)` | 파라미터 수 · 훈련 RMSE/R² · 검증 RMSE/R² 한 줄 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
try:                                    # 그림의 한글 글꼴 (Colab에서 처음 한 번 설치, 5초쯤)
    import koreanize_matplotlib
except ImportError:
    !pip -q install koreanize-matplotlib
    import koreanize_matplotlib

# ── 최소자승 도구 (강의 289–306쪽) ───────────────────────────────────────
def G_poly(x, deg, x0=0.0):
    """다항식 기저의 설계 행렬 G = [1, (x−x0), (x−x0)², …]. 열 하나 = 기저함수 하나(299쪽).
    291쪽처럼 x0를 빼 두면 차수가 높아도 계산이 덜 흔들린다."""
    x = np.atleast_1d(np.asarray(x, float))
    return np.vander(x - x0, deg + 1, increasing=True)

def G_cols(df, cols):
    """다중회귀의 설계 행렬: 첫 열은 1(상수항), 나머지 열은 측정 항목 하나씩."""
    return np.column_stack([np.ones(len(df))] + [df[c].to_numpy(float) for c in cols])

def fit(G, d):
    """최소자승 해 m = (GᵀG)⁻¹Gᵀd (305쪽). 실제 계산은 같은 답을 더 안정적으로 내는 np.linalg.lstsq."""
    return np.linalg.lstsq(G, np.asarray(d, float), rcond=None)[0]

def rmse(e):
    return float(np.sqrt(np.mean(np.square(e))))

# ── 자료 나누기: 무엇을 '빼 두고' 예측해 볼까 ───────────────────────────
def loo(n):                       # LOOCV: 하나씩 빼기 (n번 맞춘다)
    return [np.array([i]) for i in range(n)]

def kfold(n, k=5, seed=0):        # k-겹: 무작위로 k묶음
    idx = np.random.default_rng(seed).permutation(n)
    return [np.sort(f) for f in np.array_split(idx, k)]

def deepest(x, k):                # 외삽 흉내: x가 가장 큰 k개를 한꺼번에 빼기
    return [np.sort(np.argsort(np.asarray(x))[-k:])]

def cv_predict(make_G, d, folds, transform=None):
    """검증 묶음마다 '나머지 자료로만' m을 구하고, 빼 둔 자료를 예측한다.
    make_G(idx): 자료 번호 idx로 G를 만드는 함수. transform="log"이면 ln d에 맞추고 exp로 되돌린다."""
    d = np.asarray(d, float); y = np.log(d) if transform == "log" else d
    pred = np.full(len(d), np.nan)
    for va in folds:
        tr = np.setdiff1d(np.arange(len(d)), va)
        Gtr = make_G(tr)
        if Gtr.shape[1] > len(tr):    # 남은 자료 수보다 계수가 많으면 m을 정할 수 없다 → 빈칸(NaN)
            continue
        p = make_G(va) @ fit(Gtr, y[tr])
        pred[va] = np.exp(p) if transform == "log" else p
    return pred

def report(name, make_G, d, folds, transform=None):
    """모델 한 줄 요약: 파라미터 수 · 훈련 RMSE · 훈련 R² · 검증 RMSE · 검증 R²."""
    d = np.asarray(d, float); allidx = np.arange(len(d))
    G = make_G(allidx); y = np.log(d) if transform == "log" else d
    p = G @ fit(G, y); p = np.exp(p) if transform == "log" else p
    pv = cv_predict(make_G, d, folds, transform); ok = ~np.isnan(pv)
    sst = np.sum((d - d.mean()) ** 2)
    out = {"모델": name, "파라미터 수": G.shape[1],
           "훈련 RMSE": rmse(p - d), "훈련 R²": 1 - np.sum((p - d) ** 2) / sst,
           "검증 RMSE": np.nan, "검증 R²": np.nan}
    if ok.any():                      # 빼 둔 자료의 오차 (deepest처럼 일부만 빼면 그 일부로)
        out["검증 RMSE"] = rmse(pv[ok] - d[ok])
    if ok.all():
        out["검증 R²"] = 1 - np.sum((pv - d) ** 2) / sst
    return out
print("도구 준비 완료")

## 1. 1교시 연결 — 306쪽 네 점 (모두, 5분)

(2, 1), (4, 4), (6, 3), (8, 4). 1교시에 손으로 구한 **직선의 LOOCV 예측값 네 개**와 아래 출력이 같은지 확인하세요. 3차는 왜 검증 칸이 비어 있을까요?

In [ ]:
x4 = np.array([2, 4, 6, 8.]); d4 = np.array([1, 4, 3, 4.])
rows = [report(f"{deg}차", lambda i, deg=deg: G_poly(x4[i], deg), d4, loo(4)) for deg in range(4)]
t = pd.DataFrame(rows); t.insert(2, "E = eᵀe", t["훈련 RMSE"] ** 2 * 4)
display(t.round(3))
pv = cv_predict(lambda i: G_poly(x4[i], 1), d4, loo(4))
print("직선 LOOCV — 빼 둔 점의 예측값:", pv.round(3), " | 오차 d − 예측:", (d4 - pv).round(3))

## 2. 문제 1 · 공극률–깊이 — 깊이 4.5 km를 예측한다 (조 번호 ÷ 3의 나머지 = 1)

사암 시료 14개(깊이 0.24–2.96 km)의 공극률. 압밀로 공극률은 깊이에 따라 지수적으로 줄어든다고 알려져 있습니다 — **Athy 식** φ = φ₀·e^(−cz). 양변에 ln을 취하면 ln φ = ln φ₀ − c·z, 곧 **G = [1, z]인 직선 최소자승**입니다.

In [ ]:
p1 = pd.read_csv("porosity_depth.csv")
z, phi = p1["깊이_km"].to_numpy(), p1["공극률_pct"].to_numpy(); z0 = z.mean(); n1 = len(z)
plt.figure(figsize=(6, 3.8)); plt.scatter(z, phi, color="k")
plt.xlabel("깊이 (km)"); plt.ylabel("공극률 (%)"); plt.title("사암 시료 14개"); plt.show()

### 2-1. AI의 답 (기록)

- **AI가 고른 모델** (차수 · 식 · 방법):
- **AI가 보고한 성능** (숫자 그대로, **어떤 자료로 잰 값인지**도):
- **AI의 4.5 km 예측값**:
- **핵심 문장 인용** (한두 줄):

In [ ]:
# 2-2. 같은 기준에서 다시 재기 — 훈련 오차 · 검증 오차(LOOCV) · 깊은 4개를 빼고 맞춘 오차 · 4.5 km 예측
rows = []
for deg in range(1, 9):
    mk = lambda i, deg=deg: G_poly(z[i], deg, z0)
    r = report(f"{deg}차 다항식", mk, phi, loo(n1))
    r["깊은 4개 RMSE"] = report("", mk, phi, deepest(z, 4))["검증 RMSE"]
    r["4.5 km 예측"] = (G_poly(4.5, deg, z0) @ fit(mk(np.arange(n1)), phi))[0]
    rows.append(r)
mk = lambda i: G_poly(z[i], 1)                      # Athy 식: ln φ = ln φ0 − c·z  →  G = [1, z]
r = report("Athy 식 (지수)", mk, phi, loo(n1), transform="log")
r["깊은 4개 RMSE"] = report("", mk, phi, deepest(z, 4), transform="log")["검증 RMSE"]
m_athy = fit(mk(np.arange(n1)), np.log(phi)); r["4.5 km 예측"] = float(np.exp(m_athy[0] + m_athy[1] * 4.5))
rows.append(r)
t1 = pd.DataFrame(rows); display(t1.round(3))
print(f"Athy 식: φ0 = {np.exp(m_athy[0]):.1f} %,  c = {-m_athy[1]:.3f} /km")

In [ ]:
# 2-3. AI의 모델을 그림으로 — ai_deg를 AI가 고른 차수로 바꾸고 실행
ai_deg = 8                                        # ← AI가 고른 차수
zs = np.linspace(0, 5, 400)
plt.figure(figsize=(7, 4.4))
plt.axvspan(z.min(), z.max(), color="0.93", label="자료 범위")
plt.plot(zs, G_poly(zs, ai_deg, z0) @ fit(G_poly(z, ai_deg, z0), phi), color="tab:orange", label=f"AI: {ai_deg}차")
plt.plot(zs, np.exp(m_athy[0] + m_athy[1] * zs), color="tab:blue", lw=2.5, label="Athy 식")
plt.scatter(z, phi, color="k", zorder=5); plt.axvline(4.5, ls=":", color="0.4")
plt.ylim(-5, 60); plt.xlabel("깊이 (km)"); plt.ylabel("공극률 (%)"); plt.legend(); plt.show()

### 반박문 (조가 함께 쓰고, 각자 붙여 제출)

**① AI의 주장** — 핵심 문장 인용 한두 줄:

**② 무엇이 틀렸거나 빠졌나** — 다섯 질문(배포물 앞면) 중 몇 번에 걸리나:

**③ 근거** — 위 표·그림의 수치(예: 검증 RMSE ○○ vs ○○), 강의 쪽수, 지질학적 이유:

**④ 대안** — 우리가 쓸 모델과 그 이유, 그리고 그 모델의 한계:

**⑤ (선택) AI에게 이 반박문을 보여 준 뒤** — 수긍했나? 고친 답에 새로 틀린 곳은?:

*AI가 처음부터 맞게 답했다면: ②에 "다섯 질문 모두 통과", ③에 우리가 확인한 수치, ④에 AI가 말하지 않은 한계를 적는 **검증 보고**로 씁니다.*

## 3. 문제 2 · 코어 8개 — P파 속도로 밀도를 예측한다 (나머지 = 2)

퇴적암(사암·셰일·석회암) 코어 8개의 P파 속도와 밀도. 퇴적암에서는 대략 **ρ ≈ a·Vp^b** (Gardner 식, b ≈ 0.25)가 알려져 있습니다. ln을 취하면 ln ρ = ln a + b·ln Vp — 역시 **G = [1, ln Vp]인 직선 최소자승**입니다.

In [ ]:
p2 = pd.read_csv("velocity_density.csv")
v, rho = p2["Vp_km_s"].to_numpy(), p2["밀도_g_cm3"].to_numpy(); v0 = v.mean(); n2 = len(v)
plt.figure(figsize=(6, 3.8)); plt.scatter(v, rho, color="k")
plt.xlabel("P파 속도 (km/s)"); plt.ylabel("밀도 (g/cm³)"); plt.title("코어 8개"); plt.show()

def n_params(layers):          # 신경망 파라미터 수: 층마다 (들어오는 수 × 노드 수 + 노드 수)
    return sum(a * b + b for a, b in zip(layers[:-1], layers[1:]))
ai_layers = [1, 64, 64, 1]     # ← AI 신경망의 구조로 바꾸기: 입력 1 → 은닉층들 → 출력 1
print("AI 신경망의 파라미터 수:", n_params(ai_layers), " | 자료 수:", n2)

### 3-1. AI의 답 (기록)

- **AI가 만든 신경망** (층 · 노드 수 · 학습 횟수):
- **AI가 보고한 정확도** (숫자 그대로, **어떤 자료로 잰 값인지**도 — 학습에 쓴 자료? 따로 뗀 몇 개?):
- **핵심 문장 인용** (한두 줄):

In [ ]:
# 3-2. 같은 기준(LOOCV)에서 파라미터가 적은 모델들과 비교
rows = [report("평균", lambda i: np.ones((len(i), 1)), rho, loo(n2)),
        report("직선", lambda i: G_poly(v[i], 1, v0), rho, loo(n2)),
        report("Gardner형 ρ = a·Vp^b", lambda i: G_poly(np.log(v[i]), 1), rho, loo(n2), transform="log"),
        report("2차", lambda i: G_poly(v[i], 2, v0), rho, loo(n2)),
        report("3차", lambda i: G_poly(v[i], 3, v0), rho, loo(n2))]
display(pd.DataFrame(rows).round(4))

In [ ]:
# 3-3. AI의 신경망을 같은 LOOCV로 — 시드(초깃값)만 바꿔 다섯 번 (30초쯤)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
ai_hidden = tuple(ai_layers[1:-1])                 # 은닉층 크기 (위에서 고친 ai_layers를 따른다)
def mlp(seed):
    return make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=ai_hidden, max_iter=5000, random_state=seed))
res = []
for seed in range(5):
    pv = np.full(n2, np.nan)
    for va in loo(n2):
        tr = np.setdiff1d(np.arange(n2), va)
        pv[va] = mlp(seed).fit(v[tr, None], rho[tr]).predict(v[va, None])
    full = mlp(seed).fit(v[:, None], rho)
    res.append({"시드": seed, "훈련 RMSE": rmse(full.predict(v[:, None]) - rho), "검증 RMSE": rmse(pv - rho)})
display(pd.DataFrame(res).round(4))

### 반박문 (조가 함께 쓰고, 각자 붙여 제출)

**① AI의 주장** — 핵심 문장 인용 한두 줄:

**② 무엇이 틀렸거나 빠졌나** — 다섯 질문(배포물 앞면) 중 몇 번에 걸리나:

**③ 근거** — 위 표·그림의 수치(예: 검증 RMSE ○○ vs ○○), 강의 쪽수, 지질학적 이유:

**④ 대안** — 우리가 쓸 모델과 그 이유, 그리고 그 모델의 한계:

**⑤ (선택) AI에게 이 반박문을 보여 준 뒤** — 수긍했나? 고친 답에 새로 틀린 곳은?:

*AI가 처음부터 맞게 답했다면: ②에 "다섯 질문 모두 통과", ③에 우리가 확인한 수치, ④에 AI가 말하지 않은 한계를 적는 **검증 보고**로 씁니다.*

## 4. 문제 3 · 시료 16개, 측정 항목 12개 — 공극률을 예측한다 (나머지 = 0)

사암 시료 16개의 공극률과 12개 항목. 사암의 공극률을 줄이는 지질 과정은 **압밀(깊이)**, **점토가 공극을 메움**, **교결(방해석 등)**입니다. 열 설명은 배포물 뒷면 — 감마선은 점토를 따라가는 검층값이고, 광물 네 개는 합이 100 %이며, 채취월·보관일수·시료무게·코어상자는 관리용 기록입니다.

In [ ]:
p3 = pd.read_csv("sandstone_features.csv")
y3 = p3["공극률_pct"].to_numpy(); n3 = len(y3)
allc = [c for c in p3.columns if c not in ("시료", "공극률_pct")]
G_all = G_cols(p3, allc)
print("G의 크기 (행 = 시료, 열 = 상수항 + 항목):", G_all.shape, " | rank:", np.linalg.matrix_rank(G_all))
print("석영 + 장석 + 점토 + 방해석:", p3[["석영_pct", "장석_pct", "점토_pct", "방해석_pct"]].sum(axis=1).round(6).unique())
display(p3[["공극률_pct"] + allc].corr().round(2))        # 상관계수 표 (강의 118–121쪽)

### 4-1. AI의 답 (기록)

- **AI가 만든 모델** (넣은 변수 · 방법):
- **AI가 보고한 성능** (R² 숫자 그대로, **어떤 자료로 잰 값인지**도):
- **핵심 문장 인용** (한두 줄):

In [ ]:
# 4-2. 같은 기준(LOOCV)에서 변수 묶음 비교
ai_cols = allc                     # ← AI가 최종으로 넣은 변수 목록으로 바꾸기 (전부면 그대로)
my_cols = []                       # ← 우리 조가 지질학적 근거로 고른 변수 (예: ["깊이_km", ...])
sets = {"전부 12개": allc, "AI가 고른 변수": ai_cols, "우리가 고른 변수": my_cols}
rows = [report(k, lambda i, c=c: G_cols(p3.iloc[i], c), y3, loo(n3)) for k, c in sets.items()]
display(pd.DataFrame(rows).round(3))

### 반박문 (조가 함께 쓰고, 각자 붙여 제출)

**① AI의 주장** — 핵심 문장 인용 한두 줄:

**② 무엇이 틀렸거나 빠졌나** — 다섯 질문(배포물 앞면) 중 몇 번에 걸리나:

**③ 근거** — 위 표·그림의 수치(예: 검증 RMSE ○○ vs ○○), 강의 쪽수, 지질학적 이유:

**④ 대안** — 우리가 쓸 모델과 그 이유, 그리고 그 모델의 한계:

**⑤ (선택) AI에게 이 반박문을 보여 준 뒤** — 수긍했나? 고친 답에 새로 틀린 곳은?:

*AI가 처음부터 맞게 답했다면: ②에 "다섯 질문 모두 통과", ③에 우리가 확인한 수치, ④에 AI가 말하지 않은 한계를 적는 **검증 보고**로 씁니다.*

## 5. (선택) AI가 준 코드를 그대로 실행해 보기

AI의 코드를 아래 칸에 붙여 실행해 보세요. AI가 보고한 숫자가 **어떤 자료로 잰 값인지**(학습에 쓴 자료인가, 따로 뗀 자료인가, 몇 개인가)를 코드에서 찾아 반박문 ③에 씁니다.

In [ ]:
# 여기에 AI 코드를 붙여 실행하세요 (선택)

## 6. AI 활용 기록지 (2건 이상 — 오늘은 ① 처음 질의 ② 반박문을 보여 준 재질의를 권장)

| # | ① 도구 (모델·날짜) | ② 프롬프트 원문 | ③ AI 산출물 (요약·칸 번호) | ④ 검증 (무엇과 대조했나) | ⑤ 수정 (무엇을 왜) |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |